# 🚀 YOLOv8 Medium (1024px) High-Accuracy Training Pipeline

This notebook trains **YOLOv8 Medium (`yolov8m.pt`)** with **1024x1024 input resolution** on a **Google Colab Tesla T4 GPU** to achieve **higher accuracy than the YOLOv8 Nano model**.

---
### 📋 Instructions (3 Simple Steps):
1. **GPU Runtime**: Click **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. **Upload Dataset**: Drag & drop **`pipe_dataset_colab.zip`** into the Colab **Files** panel on the left.
3. **Run All**: Click **Runtime** > **Run all** (or press `Ctrl + F9`).

The notebook will train the model, benchmark accuracy, export the 1024px ONNX, and **automatically download** both files to your computer when finished!

In [ ]:
# Cell 1: Environment & GPU Verification
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime opencv-python matplotlib

import os, sys, shutil, zipfile, time, glob
import torch
from ultralytics import YOLO

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Warning: GPU not detected! Go to Runtime > Change runtime type > Select T4 GPU.")


In [ ]:
# Cell 2: Unzip Master Dataset
DATA_ZIP = "/content/pipe_dataset_colab.zip"
DATA_DIR = "/content/data"

if not os.path.exists(DATA_ZIP):
    raise FileNotFoundError("❌ /content/pipe_dataset_colab.zip not found! Please upload pipe_dataset_colab.zip to the Files tab on the left.")

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(DATA_ZIP, "r") as z:
    z.extractall(DATA_DIR)

train_n = len(os.listdir(os.path.join(DATA_DIR, "images/train")))
val_n = len(os.listdir(os.path.join(DATA_DIR, "images/val")))
print("✅ Purified Master Dataset Extracted!")
print(f"   Train images : {train_n}")
print(f"   Val images   : {val_n}")


In [ ]:
# Cell 3: Train YOLOv8 Medium (1024px) for Peak Accuracy
# Starts from yolov8m.pt base architecture (25.8 million parameters)
model = YOLO('yolov8m.pt')

TRAIN_CONFIG = {
    "data": "/content/data/data.yaml",
    "epochs": 60,               # Deep multi-epoch convergence
    "patience": 20,              # Ample patience to pass loss plateaus
    "imgsz": 1024,               # 1024x1024 high resolution
    "batch": 8,                  # Optimal for T4 16GB VRAM
    "device": 0,                 # CUDA GPU
    "optimizer": "AdamW",        # Adaptive momentum
    "lr0": 0.0003,               # Stable fine-tuning rate
    "lrf": 0.01,                 # Cosine learning rate floor
    "cos_lr": True,              # Smooth cosine annealing scheduler
    "box": 7.5,                  # High bounding box loss weight
    "cls": 1.5,                  # Suppresses background false positives
    "dfl": 1.5,                  # Distribution focal loss
    "degrees": 15.0,             # Rotational invariance for angled pipes
    "translate": 0.1,            # Translation jitter
    "scale": 0.6,                # Scale invariance (near vs far)
    "fliplr": 0.5,               # Horizontal flip
    "flipud": 0.2,               # Vertical flip
    "mosaic": 1.0,               # 4-image mosaic for dense pipe scenes
    "mixup": 0.15,               # Mixup for overlapping pipes
    "close_mosaic": 15,          # Disables mosaic during final 15 epochs for clean natural edges
    "save": True,
    "project": "pipe_training",
    "name": "yolov8m_1024_accuracy"
}

print("="*70)
print("🚀 STARTING YOLOV8M (1024px) DEEP TRAINING ON GPU")
print("="*70)
t0 = time.time()
results = model.train(**TRAIN_CONFIG)
elapsed = (time.time() - t0) / 60.0
print(f"\n✅ Training completed in {elapsed:.1f} minutes!")


In [ ]:
# Cell 4: Validation & Accuracy Metrics Evaluation
best_weights = glob.glob("/content/pipe_training/yolov8m_1024_accuracy*/weights/best.pt")[-1]
print(f"🎯 Evaluating Best Model: {best_weights}")
best_model = YOLO(best_weights)

metrics = best_model.val(data="/content/data/data.yaml", imgsz=1024, device=0)

p = metrics.box.p[0] * 100.0
r = metrics.box.r[0] * 100.0
map50 = metrics.box.map50 * 100.0
map95 = metrics.box.map * 100.0
f1 = 2 * (p * r) / (p + r + 1e-16)

print("\n" + "="*60)
print("🎯 FINAL YOLOV8M VALIDATION ACCURACY:")
print(f"   Precision (P) : {p:.2f}%")
print(f"   Recall (R)    : {r:.2f}%")
print(f"   mAP@50        : {map50:.2f}%")
print(f"   mAP@50-95     : {map95:.2f}%")
print(f"   F1-Score      : {f1:.2f}%")
print("="*60)


In [ ]:
# Cell 5: Multi-Scenario Benchmark Suite
candidates = [
    ("70-Pipe Grid Carton", "/content/data/sample_test/box_touching_grid_70.jpg", 0.35, 70),
    ("Multi-Layer Stack", "/content/data/sample_test/pipe_stack_dreamstime.jpg", 0.28, 29),
    ("Warehouse Box Carton", "/content/data/sample_test/warehouse_box_uploaded.jpg", 0.50, 58),
    ("6-Pipe Pyramid Bundle", "/content/data/sample_test/crop_6_pipes.jpg", 0.28, 6),
    ("Loose Deformed Pipes", "/content/data/sample_test/box_deformed_pipes_clean.jpg", 0.35, 48),
]

print("="*70)
print("🧪 MULTI-SCENARIO BENCHMARK VERIFICATION")
print("="*70)
for title, fpath, conf_val, expected in candidates:
    if os.path.exists(fpath):
        res = best_model.predict(source=fpath, conf=conf_val, imgsz=1024, verbose=False)[0]
        det_count = len(res.boxes)
        acc_pct = (1.0 - abs(det_count - expected) / float(expected)) * 100.0
        print(f"  {title:25s} | Detected: {det_count:3d} / {expected:3d} ({acc_pct:.1f}% accuracy)")
    else:
        print(f"  {title:25s} | File not found at {fpath}")
print("="*70)


In [ ]:
# Cell 6: Export to 1024px ONNX & Download Both Models
from google.colab import files

print("⚡ Exporting refined model to 1024px ONNX...")
onnx_path = best_model.export(format="onnx", imgsz=1024, simplify=True)

pt_dst = "/content/best_insulation_pipe_m_1024.pt"
onnx_dst = "/content/best_insulation_pipe_m_1024.onnx"

shutil.copy(best_weights, pt_dst)
shutil.copy(onnx_path, onnx_dst)

print("\n⬇️ Downloading best_insulation_pipe_m_1024.pt (PyTorch weights, ~52 MB)...")
files.download(pt_dst)

print("⬇️ Downloading best_insulation_pipe_m_1024.onnx (1024px ONNX engine, ~104 MB)...")
files.download(onnx_dst)

print("\n🎉 Training Complete! Move the 2 downloaded files into c:\\AI\\backend\\models\\ on your PC!")
